# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [3]:
# Clean consumer transactions
import pandas as pd
import numpy as np


def generate_transactions(n_orders=20000, random_seed=42):

    # Fix the random seed
    np.random.seed(random_seed)

    # Randomly assign
    order_year = np.random.choice([2020, 2021], size=n_orders)

    # Cancellations in 2021= 10%, nothing cancelled in 2020
    cancel_flag = np.where(
        order_year == 2021,
        np.random.choice([0, 1], size=n_orders, p=[0.9, 0.1]),
        0
    )

    # label every order as consumer (B2C)
    customer_segment = np.array(["B2C"] * n_orders)

    # Pick 20 random orders and label as business (B2B)
    b2b_positions = np.random.choice(n_orders, size=20, replace=False)
    customer_segment[b2b_positions] = "B2B"

    # log-normal distribution
    spend_amount = np.random.lognormal(
        mean=3.5,
        sigma=0.8,
        size=n_orders
    )

    # 100x the basket
    spend_amount[b2b_positions] *= 100

    # build df
    columns = {
        "basket_value": spend_amount,
        "year": order_year,
        "order_type": customer_segment,
        "customer_id": np.random.choice(1000, size=n_orders),
        "cancelled": cancel_flag,
    }

    return pd.DataFrame(columns)


transactions_df = generate_transactions()

In [4]:
# The truth: the consumer average per year, recorded before contamination
def true_mean_by_year(transactions):
    # Keep non cancelled, then average by year
    clean_2020 = transactions.loc[
        (transactions["year"] == 2020) &
        (transactions["order_type"] == "B2C") &
        (transactions["cancelled"] == 0),
        "basket_value"
    ].mean()

    clean_2021 = transactions.loc[
        (transactions["year"] == 2021) &
        (transactions["order_type"] == "B2C") &
        (transactions["cancelled"] == 0),
        "basket_value"
    ].mean()
    return [clean_2020, clean_2021]


true_results = true_mean_by_year(transactions_df)
print(f"The true mean of 2020 and 2021: {true_results[0]}, {true_results[1]}")

The true mean of 2020 and 2021: 45.97798987067258, 45.71762641987248


### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [5]:
#dashboard's number vs the truth
def naive_mean_by_year(transactions):
   #all orders included
    rows_2021 = transactions.loc[transactions["year"] == 2021]
    rows_2020 = transactions.loc[transactions["year"] == 2020]

    return [
        rows_2020["basket_value"].mean(),
        rows_2021["basket_value"].mean()
    ]


def mean_gap_by_year(transactions):
   #naive mean vs true mean
    true_vals = true_mean_by_year(transactions)
    naive_vals = naive_mean_by_year(transactions)
    return [abs(true_vals[0] - naive_vals[0]), abs(true_vals[1] - naive_vals[1])]


naive_results = naive_mean_by_year(transactions_df)
gap_results = mean_gap_by_year(transactions_df)
print(f"The naive mean of 2020 and 2021: {naive_results[0]}, {naive_results[1]}")
print(f"The true mean of 2020 and 2021: {true_results[0]}, {true_results[1]}")
print(f"The gap between the true and naive mean for 2020 and 2021: {gap_results[0]}, {gap_results[1]}")

The naive mean of 2020 and 2021: 49.3380734922399, 51.60815246087797
The true mean of 2020 and 2021: 45.97798987067258, 45.71762641987248
The gap between the true and naive mean for 2020 and 2021: 3.3600836215673198, 5.8905260410054865


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [6]:
# How much of the gap is the skew and the B2B tail?
def split_gap_sources(transactions):

    true_vals = true_mean_by_year(transactions)
    naive_vals = naive_mean_by_year(transactions)

    # Mean after removing cancelled orders only
    noncancelled_mean_2020 = transactions.loc[
        (transactions["year"] == 2020) & (transactions["cancelled"] == 0),
        "basket_value"
    ].mean()

    noncancelled_mean_2021 = transactions.loc[
        (transactions["year"] == 2021) & (transactions["cancelled"] == 0),
        "basket_value"
    ].mean()

    b2b_impact = [
        noncancelled_mean_2020 - true_vals[0],
        noncancelled_mean_2021 - true_vals[1]
    ]

    logging_impact = [
        naive_vals[0] - noncancelled_mean_2020,
        naive_vals[1] - noncancelled_mean_2021
    ]

    return [b2b_impact, logging_impact]


gap_sources = split_gap_sources(transactions_df)
print(f"B2B effect for 2020 and 2021: {gap_sources[0]}")

B2B effect for 2020 and 2021: [np.float64(3.3600836215673198), np.float64(6.2983883259727875)]


In [7]:
# How much is the change in what was logged?
print(f"Logging effect for 2020 and 2021: {gap_sources[1]}")

Logging effect for 2020 and 2021: [np.float64(0.0), np.float64(-0.40786228496730104)]


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail | 6.3 | (cancellations removed, B2B kept) - (fully clean mean) |
| Change in what was logged | -0.4 | naive mean - (cancellations removed, B2B kept) |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [8]:
# Median, trimmed mean, and the mean after your B2B exclusion rule
def median_by_year(transactions):

    rows_2021 = transactions.loc[transactions["year"] == 2021]
    rows_2020 = transactions.loc[transactions["year"] == 2020]
    return [rows_2020["basket_value"].median(), rows_2021["basket_value"].median()]


def trimmed_mean_by_year(transactions, trim_share):

    rows_2021 = transactions.loc[transactions["year"] == 2021]
    rows_2020 = transactions.loc[transactions["year"] == 2020]

    yearly_slices = [rows_2020, rows_2021]
    trimmed_results = []

    for yearly_rows in yearly_slices:
        # Sort basket values from smallest to largest
        ordered_values = sorted(yearly_rows["basket_value"])
        n_rows = len(ordered_values)

        # Number of observations to cut from each end
        n_cut = int(n_rows * trim_share)

        if n_cut > 0:
            kept_values = ordered_values[n_cut:-n_cut]
        else:
            kept_values = ordered_values

        # Guard against an empty list (would otherwise divide by zero)
        if not kept_values:
            trimmed_results.append(0)
        else:
            trimmed_results.append(sum(kept_values) / len(kept_values))

    return trimmed_results


def mean_excluding_b2b(transactions):

    consumer_mean_2020 = transactions.loc[
        (transactions["year"] == 2020) & (transactions["order_type"] == "B2C"),
        "basket_value"
    ].mean()

    consumer_mean_2021 = transactions.loc[
        (transactions["year"] == 2021) & (transactions["order_type"] == "B2C"),
        "basket_value"
    ].mean()
    return [consumer_mean_2020, consumer_mean_2021]


print(f"The median basket value for 2020 and 2021: {median_by_year(transactions_df)[0]}, {median_by_year(transactions_df)[1]}")
print(f"The 10% trimmed mean for 2020 and 2021: {trimmed_mean_by_year(transactions_df, 0.1)[0]}, {trimmed_mean_by_year(transactions_df, 0.1)[1]}")
print(f"The mean after B2B exclusion for 2020 and 2021: {mean_excluding_b2b(transactions_df)[0]}, {mean_excluding_b2b(transactions_df)[1]}")

The median basket value for 2020 and 2021: 33.28938209267342, 33.38633255431257
The 10% trimmed mean for 2020 and 2021: 38.457619507128555, 38.32621117129096
The mean after B2B exclusion for 2020 and 2021: 45.97798987067258, 45.74759576489542


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median | 33.38 | You want the typical basket, and outliers shouldn't pull the result. | you're looking at total revenue |
| Trimmed mean | 38.32 | Outliers are rare, roughly symmetric, and not real purchases. | Extremes are real and drive revenue, or the skew is one-sided |
| Mean after B2B exclusion | 45.74 | B2B is the only effect, and cancelled orders are negligible | Cancelled orders distort baskets, or B2C has its own heavy tail. |

**Which number should the dashboard show?** Commit to one and defend it.

I would use the mean after the B2B exlusion because it is the most accurate measure of the consumer basket. This mean value gets rid of the B2B sales that were not meant to be in the dataset. It also keeps the real consumer data points like the trimmed mean.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [9]:
import os
os.makedirs("src", exist_ok=True)

In [10]:
%%writefile src/basket_metrics.py
"""Audit and robust-average helpers for the consumer basket-value metric."""

import numpy as np
import pandas as pd


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
  """One row per column: missingness, dtype, skew and outlier count."""

  report_rows = []
  for col_name in df:
    col_values = df[col_name]

    #rows that are missing (NaN) in this column
    share_missing = col_values.isna().mean()

    col_type = col_values.dtype

    if (pd.api.types.is_numeric_dtype(col_values)):

        col_skew = col_values.skew()

    #find spread (IQR)
        first_quartile = col_values.quantile(0.25)
        third_quartile = col_values.quantile(0.75)
        spread = third_quartile - first_quartile

        # Values more than 1.5 IQRs outside the quartiles count as outliers
        floor_cutoff = first_quartile - 1.5 * spread
        ceiling_cutoff = third_quartile + 1.5 * spread

        n_outliers = (
            (col_values < floor_cutoff) |
            (col_values > ceiling_cutoff)
        ).sum()

    else:
        #no outliers or text column
        col_skew = np.nan
        n_outliers = np.nan

    report_rows.append({
        "column": col_name,
        "missing": share_missing,
        "dtype": col_type,
        "skew": col_skew,
        "outlier_count": n_outliers
    })
  return pd.DataFrame(report_rows)


def robust_mean(
    x: pd.Series,
    method: str = "median",
    segment: pd.Series | None = None
    ) -> float:

  match method:
    case "median":
      return x.median()

    case "trimmed":
      # Keep only values between the 10th and 90th percentiles, then average
      low_cut = x.quantile(0.1)
      high_cut = x.quantile(0.9)

      return x.loc[
          (x >= low_cut) & (x <= high_cut)
      ].mean()

    case "rule-based":
      # Needs the order-type label
      if segment is None:
        raise ValueError("order_type must be provided")
      return x.loc[segment == "B2C"].mean()

    case _:
            raise ValueError(
                "The method is not valid. "
                "Use 'median', 'trimmed', or 'rule-based'."
            )

if __name__ == "__main__":
    # A short demonstration that runs when the file is run as a script
    demo_df = pd.DataFrame({
        "basket_value": [10, 12, 15, 20, 25, 30, 1000],
        "order_type": ["B2C", "B2C", "B2C", "B2C", "B2C", "B2C", "B2B"]
    })

    print("Audit Report:")
    print(audit_report(demo_df))

    print("\nMedian:")
    print(robust_mean(demo_df["basket_value"], "median"))

    print("\nTrimmed Mean:")
    print(robust_mean(demo_df["basket_value"], "trimmed"))

    print("\nRule-based Exclusion:")
    print(robust_mean(demo_df["basket_value"], "rule-based", demo_df["order_type"]))


Writing src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [11]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)     # picks up your edits when you re-run %%writefile

# Tiny test dataset: nine ordinary consumer baskets plus one huge B2B basket
sample_df = pd.DataFrame({
    "basket_value": [10, 12, 15, 18, 20, 22, 25, 30, 35, 1000],
    "order_type": [
        "B2C", "B2C", "B2C", "B2C", "B2C",
        "B2C", "B2C", "B2C", "B2C", "B2B"
    ],
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
})

# each average on the test data
median_result = basket_metrics.robust_mean(sample_df["basket_value"], "median")
trimmed_result = basket_metrics.robust_mean(sample_df["basket_value"], "trimmed")
rule_based_result = basket_metrics.robust_mean(sample_df["basket_value"], "rule-based", sample_df["order_type"])

print("Audit Report:")
print(basket_metrics.audit_report(sample_df))

print("robust_mean: median")
print(median_result)

print("robust_mean: trimmed")
print(trimmed_result)

print("robust_mean: rule based")
print(rule_based_result)

# Automatic checks against hand-computed values
assert median_result == 21
assert rule_based_result == 20.77777777777778
assert trimmed_result < sample_df["basket_value"].mean()

# At least three properties that must hold, for example:
# assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < tolerance

Audit Report:
         column  missing   dtype      skew  outlier_count
0  basket_value      0.0   int64  3.158567            1.0
1    order_type      0.0  object       NaN            NaN
2   customer_id      0.0   int64  0.000000            0.0
robust_mean: median
21.0
robust_mean: trimmed
22.125
robust_mean: rule based
20.77777777777778


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**

In [12]:
# The analysis that settles it
def build_bridge(transactions: pd.DataFrame, trim_share: float = 0.1) -> pd.DataFrame:
    """Reconcile the dashboard mean to the true consumer mean, one step at a time.

    Steps run in a fixed order, so each contribution is exact and the pieces
    sum to the total gap: naive -> drop B2B -> drop cancelled -> true mean.
    The trimmed mean is NOT a step in the bridge. It is reported afterwards as
    a sensitivity, because it does not estimate the consumer mean.
    """
    rows = []
    for year, g in transactions.groupby("year"):
        naive = g["basket_value"].mean()                                    # dashboard
        no_b2b = g.loc[g["order_type"] == "B2C", "basket_value"].mean()     # step 1: B2B out
        harmonized = g.loc[(g["order_type"] == "B2C") & (g["cancelled"] == 0),
                           "basket_value"].mean()                           # step 2: same logging rule both years
        trimmed = stats.trim_mean(
            g.loc[(g["order_type"] == "B2C") & (g["cancelled"] == 0), "basket_value"],
            trim_share)                                                     # sensitivity only
        rows.append({
            "year": year,
            "naive_mean": naive,
            "b2b_contribution": naive - no_b2b,
            "logging_contribution": no_b2b - harmonized,
            "true_consumer_mean": harmonized,
            "total_gap": naive - harmonized,
            "trimmed_mean": trimmed,
            "trim_bias_vs_true": trimmed - harmonized,
        })
    bridge = pd.DataFrame(rows)

    # The bridge must close: naive - contributions = true mean, within rounding
    closes = np.isclose(bridge["naive_mean"] - bridge["b2b_contribution"]
                        - bridge["logging_contribution"], bridge["true_consumer_mean"])
    assert closes.all(), "bridge does not reconcile"
    return bridge

bridge = build_bridge(transactions_df)
bridge.round(2)

,year,naive_mean,b2b_contribution,logging_contribution,true_consumer_mean,total_gap,trimmed_mean,trim_bias_vs_true
0,2020,49.34,3.36,0.00,45.98,3.36,38.43,-7.55
1,2021,51.61,5.86,0.03,45.72,5.89,38.19,-7.53




```
# This is formatted as code
```

**What you found, including if it changes your Phase 2 recommendation:**
The bridge matches Phase 1: almost all of the 2021 gap (5.86 of 5.89) comes from B2B orders, and the logging change adds almost nothing. Once B2B is removed, the consumer basket is flat from 2020 to 2021, so the dashboard's rise was mostly B2B. The trimmed mean ends up about 7.5 too low, so my Phase 2 recommendation stays the same: use the mean after B2B exclusion, and also remove cancelled orders.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

## The dashboard overstated the average consumer basket by **\$3.36 in 2020** and **\$5.89 in 2021**.

## The Correction
- Almost all of the gap comes from B2B orders (\$5.86 of \$5.89 in 2021)
- The logging change adds almost nothing (\$0.03)
- After the fix, the consumer basket is **flat**: \$45.98 (2020) to \$45.72 (2021)
- The dashboard's apparent rise was mostly B2B

## Recommendation
- Show the mean after B2B exclusion, with cancelled orders removed in both years
- Do not use the trimmed mean: it runs about \$7.50 too low

## Before We Bet On It
Confirm that B2B orders and cancellations are labeled correctly in the real data.